# Level-1 reproduction in the browser（ブラウザで水準 1 の再現）

上から順に **▶ を押すだけ**です（所要 5〜10 分・GPU 不要）。最後のセルの出力（`report.txt` の中身）をコピーして送ってください。

- 水準 1 ＝ コミット済みの結果 CSV から論文の図と表を作り直し、原稿の全表のセルが一致するかを `check_tables.py` で機械的に確かめる
- 個人情報は取りません。出力に含まれるのは OS・Python・ライブラリのバージョンと、テスト・図・表の結果だけです
- 途中で止まったら、そのセルのエラーを含めて出力を送ってください（それも重要な結果です）

In [ ]:
%%bash
set -e
rm -rf qaoa-portfolio-penalty-benchmark
git clone -q https://github.com/tobitaQ/qaoa-portfolio-penalty-benchmark.git
cd qaoa-portfolio-penalty-benchmark
git checkout -q v1.0
echo "commit: $(git log --oneline -1)"
python --version
pip install -q -r requirements.txt 2>&1 | tail -2
echo "install: done"

In [ ]:
%%bash
cd qaoa-portfolio-penalty-benchmark
python -m pytest -q 2>&1 | tail -3

In [ ]:
%%bash
cd qaoa-portfolio-penalty-benchmark
python -m notebooks.generate_tqe_figures 2>&1 | tail -3
cd papers && python check_tables.py 2>&1 | tail -8

In [ ]:
%%bash
cd qaoa-portfolio-penalty-benchmark
{
  echo "== package"; git log --oneline -1
  echo "== python"; python --version
  pip freeze 2>/dev/null | grep -iE "^(pennylane|pennylane[-_]lightning|numpy|pandas|matplotlib|scipy)==" || true
  echo "== tests"; python -m pytest -q 2>&1 | tail -1
  echo "== level 1"; ( cd papers && python check_tables.py 2>&1 | tail -4 )
} > report.txt
cat report.txt
echo
echo "↑ この出力をコピーし、README の報告フォームに貼ってください（環境の欄はご自身で選んで記入）"


（任意）図の比較。パッケージには論文の図がコミットされていて、上のセルはそれを同じ名前で上書きしています。各図について、論文の版と再生成版を**ピクセル単位で比較した数字**（異なる画素の割合）を出し、その下に左＝論文の版・右＝再生成版を並べます。0 % なら同一、数 % 以内なら Matplotlib の版差による描画の揺れです（数字の判定は表の一致数で済んでいるので、これは参考）。論文の Fig. 1〜4 は `fig_e0_rescore`・`fig_e1_penalty`・`fig_e2_shots`・`fig_e4_backends`（`fig1_…` などの番号付きは補足資料の図）：

In [ ]:
import subprocess, pathlib, base64, io
import numpy as np
from PIL import Image as PILImage
from IPython.display import display, HTML
repo = pathlib.Path("qaoa-portfolio-penalty-benchmark")
main = [("Fig. 1", "fig_e0_rescore"), ("Fig. 2", "fig_e1_penalty"), ("Fig. 3", "fig_e2_shots"), ("Fig. 4", "fig_e4_backends")]
def b64(data): return "data:image/png;base64," + base64.b64encode(data).decode()
rows = []
for label, name in main:
    paper = subprocess.run(["git", "show", f"HEAD:notebooks/figures/{name}.png"], cwd=repo, capture_output=True).stdout
    regen = (repo / "notebooks/figures" / f"{name}.png").read_bytes()
    a = np.asarray(PILImage.open(io.BytesIO(paper)).convert("RGB")); b = np.asarray(PILImage.open(io.BytesIO(regen)).convert("RGB"))
    if paper == regen: verdict = "バイト単位で同一"
    elif a.shape == b.shape: verdict = f"同じ大きさ {a.shape[1]}×{a.shape[0]}、異なる画素 {100*np.mean(np.any(a != b, axis=2)):.2f} %"
    else: verdict = f"大きさが違う: 論文 {a.shape[1]}×{a.shape[0]} / 再生成 {b.shape[1]}×{b.shape[0]}"
    print(f"{label} {name}: {verdict}")
    rows.append(f"<tr><td colspan=2><b>{label} — {name}</b>：{verdict}</td></tr>"
                f"<tr><td style='text-align:center'>論文の版<br><img src='{b64(paper)}' style='width:100%'></td>"
                f"<td style='text-align:center'>再生成版<br><img src='{b64(regen)}' style='width:100%'></td></tr>")
display(HTML("<table style='width:100%;table-layout:fixed'>" + "".join(rows) + "</table>"))
